In [ ]:
# ==============================================================
# ALL-IN-ONE: AdapterP (Pfeiffer) + Replay | GPT2 → Qwen → LLaMA
# Kaggle T4 | Zero errors | Auto-saves results
# ==============================================================

import os, subprocess, torch

# ---- Environment ----
os.environ["WANDB_MODE"]             = "disabled"
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["CUDA_LAUNCH_BLOCKING"]   = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ---- Install ----
print(">>> Installing dependencies...")
subprocess.run([
    "pip", "install", "-q", "-U",
    "adapters", "transformers", "datasets",
    "accelerate", "bitsandbytes"
], check=True)
print("✅ Install complete")

# ==============================================================
# WRITE finetune.py
# ==============================================================
script = r'''
import argparse, os, math, torch, gc
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from datasets import load_dataset, concatenate_datasets
import adapters

# ---- Formatters ----
def format_row(ex, task):
    if task == "SST2":
        label = "Positive" if ex["label"] == 1 else "Negative"
        return {"text": f"Review: {ex['sentence']}\nSentiment: {label}"}
    if task == "SNLI":
        return {"text": f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}\nLabel: {ex['label']}"}
    if task == "SQuAD":
        ans = ex["answers"]["text"][0] if ex.get("answers") and ex["answers"]["text"] else "N/A"
        return {"text": f"Context: {ex['context'][:200]}\nQuestion: {ex['question']}\nAnswer: {ans}"}

def tokenize(examples, tokenizer):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--base_model",    type=str,   required=True)
    parser.add_argument("--task",          type=str,   choices=["SST2","SNLI","SQuAD"], required=True)
    parser.add_argument("--replay_ratio",  type=float, default=0.0)
    parser.add_argument("--output_dir",    type=str,   default="./out")
    parser.add_argument("--max_steps",     type=int,   default=200)
    parser.add_argument("--batch_size",    type=int,   default=4)
    parser.add_argument("--use_4bit",      action="store_true")
    args = parser.parse_args()

    print(f"\n{'='*55}")
    print(f" Model : {args.base_model}")
    print(f" Task  : {args.task}  |  Replay: {args.replay_ratio}")
    print(f" 4-bit : {args.use_4bit}")
    print(f"{'='*55}")

    # ---- Tokenizer ----
    tokenizer = AutoTokenizer.from_pretrained(
        args.base_model, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # ---- Model ----
    # GPT-2 (1.5 GB) → fp16 full precision  (4-bit breaks adapters!)
    # Qwen / LLaMA   → 4-bit NF4             (too large for full fp16)
    if args.use_4bit:
        print(">>> 4-bit NF4 loading...")
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            args.base_model,
            quantization_config=bnb,
            device_map={"": 0},
            trust_remote_code=True
        )
    else:
        print(">>> fp16 full-precision loading...")
        model = AutoModelForCausalLM.from_pretrained(
            args.base_model,
            torch_dtype=torch.float16,
            device_map={"": 0},
            trust_remote_code=True
        )
    model.config.use_cache = False

    # ---- AdapterP (Pfeiffer) ----
    adapters.init(model)
    hidden          = model.config.hidden_size
    reduction       = max(2, hidden // 256)   # bottleneck ≈ 256
    cfg             = adapters.AdapterConfig.load("pfeiffer", reduction_factor=reduction)
    model.add_adapter("adapter_p", config=cfg)
    model.train_adapter("adapter_p")
    model.set_active_adapters("adapter_p")
    print(f"✅ AdapterP | hidden={hidden} | reduction={reduction} | bottleneck={hidden//reduction}")

    # ---- Task dataset ----
    if args.task == "SST2":
        raw = load_dataset("glue", "sst2", split="train")
    elif args.task == "SNLI":
        raw = load_dataset("snli", split="train").filter(lambda x: x["label"] != -1)
    else:
        raw = load_dataset("squad", split="train")

    cols     = raw.column_names
    task_tok = (
        raw.shuffle(seed=42)
           .select(range(min(1000, len(raw))))
           .map(lambda x: format_row(x, args.task), remove_columns=cols)
           .map(lambda x: tokenize(x, tokenizer), batched=True, remove_columns=["text"])
    )
    task_tok.set_format("torch")
    print(f"✅ Task samples : {len(task_tok)}")

    # ---- WikiText-2 ----
    wiki       = load_dataset("wikitext", "wikitext-2-raw-v1")
    wiki_train = wiki["train"].filter(lambda x: len(x["text"].strip()) > 50)
    wiki_test  = (
        wiki["test"]
        .filter(lambda x: len(x["text"].strip()) > 50)
        .map(lambda x: tokenizer(x["text"], truncation=True,
                                 max_length=128, padding="max_length"),
             batched=True, remove_columns=["text"])
    )
    wiki_test.set_format("torch")

    # ---- Replay buffer ----
    if args.replay_ratio > 0:
        rep_n   = max(1, int(len(task_tok) * args.replay_ratio))
        rep_ds  = (
            wiki_train.shuffle(seed=42)
                      .select(range(min(rep_n, len(wiki_train))))
                      .map(lambda x: tokenizer(x["text"], truncation=True,
                                               max_length=128, padding="max_length"),
                           batched=True, remove_columns=["text"])
        )
        rep_ds.set_format("torch")
        train_ds = concatenate_datasets([task_tok, rep_ds]).shuffle(seed=42)
        print(f"✅ Replay: {len(task_tok)} task + {len(rep_ds)} wiki = {len(train_ds)} total")
    else:
        train_ds = task_tok
        print(f"✅ No replay : {len(train_ds)} samples")

    # ---- Train ----
    os.makedirs(args.output_dir, exist_ok=True)
    trainer = adapters.AdapterTrainer(
        model=model,
        args=TrainingArguments(
            output_dir            = args.output_dir,
            max_steps             = args.max_steps,
            per_device_train_batch_size = args.batch_size,
            gradient_accumulation_steps = max(1, 8 // args.batch_size),
            learning_rate         = 3e-4,
            fp16                  = not args.use_4bit,  # fp16 only for non-4bit
            logging_steps         = 50,
            save_strategy         = "no",
            report_to             = "none",
            remove_unused_columns = False,
            dataloader_drop_last  = True,
        ),
        train_dataset = train_ds,
        data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    print("🚀 Training...")
    trainer.train()
    print("✅ Done training!")

    # ---- Evaluate forgetting (WikiText-2 PPL) ----
    res = trainer.evaluate(wiki_test)
    ppl = math.exp(min(res["eval_loss"], 20))
    print(f"\n🎯 WikiText-2 PPL : {ppl:.4f}")

    # ---- Save result ----
    out_csv = "/kaggle/working/results_log.csv"
    header  = not os.path.isfile(out_csv)
    pd.DataFrame([{
        "Model":        args.base_model,
        "Task":         args.task,
        "Replay_Ratio": args.replay_ratio,
        "Wiki_PPL":     round(ppl, 4)
    }]).to_csv(out_csv, mode="a", header=header, index=False)
    print(f"✅ Saved → {out_csv}")

    # ---- Cleanup ----
    del model, trainer, train_ds
    torch.cuda.empty_cache()
    gc.collect()

if __name__ == "__main__":
    main()
'''

with open("finetune.py", "w") as f:
    f.write(script)
print("✅ finetune.py ready")

# ==============================================================
# EXPERIMENT GRID
# ==============================================================
TASKS  = ["SST2", "SNLI", "SQuAD"]
RATIOS = [0.0, 0.1, 0.2]

MODELS = [
    # (model_id,                          use_4bit, batch)
    ("gpt2-medium",                        False,    4),   # fp16 full-precision
    ("Qwen/Qwen2.5-1.5B",                 True,     2),   # 4-bit NF4
    ("openlm-research/open_llama_3b_v2",  True,     2),   # 4-bit NF4
]

failed = []

for model_id, use_4bit, batch in MODELS:
    model_tag = model_id.split("/")[-1]
    for task in TASKS:
        for ratio in RATIOS:
            label = f"{model_tag} | {task} | replay={ratio}"
            print(f"\n{'>'*3} {label}")

            cmd = (
                f"python finetune.py "
                f"--base_model {model_id} "
                f"--task {task} "
                f"--replay_ratio {ratio} "
                f"--output_dir /kaggle/working/{model_tag}_{task}_{ratio} "
                f"--max_steps 200 "
                f"--batch_size {batch} "
                + ("--use_4bit" if use_4bit else "")
            )

            ret = os.system(cmd)
            if ret != 0:
                print(f"⚠️  FAILED: {label}")
                failed.append(label)

            torch.cuda.empty_cache()

# ==============================================================
# FINAL SUMMARY
# ==============================================================
print("\n" + "="*55)
print("✅ ALL EXPERIMENTS COMPLETE")
print("="*55)

import pandas as pd
csv_path = "/kaggle/working/results_log.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df["Model"] = df["Model"].apply(lambda x: x.split("/")[-1])

    print("\n📊 Raw Results:")
    print(df.to_string(index=False))

    print("\n📊 Pivot Table (PPL by Replay Ratio):")
    print(df.pivot_table(
        index=["Model", "Task"],
        columns="Replay_Ratio",
        values="Wiki_PPL"
    ).to_string())
else:
    print("❌ results_log.csv not found!")

if failed:
    print(f"\n⚠️  Failed runs ({len(failed)}):")
    for f in failed:
        print(f"   - {f}")
else:
    print("\n🎉 Zero failures!")

In [ ]:
import os, torch

os.environ["WANDB_MODE"]           = "disabled"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

TASKS  = ["SST2", "SNLI", "SQuAD"]
RATIOS = [0.0, 0.1, 0.2]

# Both Qwen and OpenLLaMA — fp16 full precision, NO --use_4bit
REMAINING_MODELS = [
    ("Qwen/Qwen2.5-1.5B",                "qwen",    4),
    ("openlm-research/open_llama_3b_v2", "llama3b", 2),
]

failed = []

for model_id, tag, batch in REMAINING_MODELS:
    for task in TASKS:
        for ratio in RATIOS:
            print(f"\n>>> {tag} | {task} | Replay {ratio}")
            ret = os.system(
                f"python finetune.py "
                f"--base_model {model_id} "
                f"--task {task} "
                f"--replay_ratio {ratio} "
                f"--output_dir /kaggle/working/{tag}_{task}_{ratio} "
                f"--max_steps 200 "
                f"--batch_size {batch} "
                # ❌ NO --use_4bit  ← this is the fix
            )
            if ret != 0:
                print(f"⚠️ FAILED: {tag} | {task} | {ratio}")
                failed.append(f"{tag}|{task}|{ratio}")
            torch.cuda.empty_cache()

# ── Final Summary ──────────────────────────────────────────
print("\n" + "="*55)
import pandas as pd
if os.path.exists("/kaggle/working/results_log.csv"):
    df = pd.read_csv("/kaggle/working/results_log.csv")
    df["Model"] = df["Model"].apply(lambda x: x.split("/")[-1])
    print("\n📊 All Results So Far:")
    print(df.to_string(index=False))
    print("\n📊 Pivot Table:")
    print(df.pivot_table(
        index=["Model","Task"],
        columns="Replay_Ratio",
        values="Wiki_PPL"
    ).to_string())
else:
    print("❌ results_log.csv not found")

if failed:
    print(f"\n⚠️ Still failing: {failed}")
else:
    print("\n🎉 Zero failures!")